In [1]:

from time import sleep
from glob import glob

# PROJECT IMPORTS
import sys
sys.path.append('..')
from setup import reddit
from settings import *

# THIRD PARTY IMPORTS
import praw
import pandas as pd

subreddit = reddit.subreddit( TARGET_SUBREDDIT )

Version 7.5.0 of praw is outdated. Version 7.8.1 was released Friday October 25, 2024.


Loaded 42 users who have opted out from follow-ups.
Loaded 30 users who have opted out from reminders.


In [6]:
# load users who get 'Star Critiquer' flair.
bonus_flair_users = pd.read_csv('/projects/bdata/photocritique-coach-extension/logs/finished_flaired_participants.csv')
bonus_flair_users.time_finished = pd.to_datetime(bonus_flair_users.time_finished, unit='s', utc=True)
# update the flair of people who got added to the list in the last 48 hours
bonus_flair_users['force_update'] = pd.Timestamp.utcnow()-bonus_flair_users.time_finished < pd.Timedelta(hours=48)

bonus_flair_set = set(bonus_flair_users.username.to_list())
        
print(f'Loaded {len(bonus_flair_users)} users who get bonus flair for participating in the Coach user study.')


# load points
df = pd.concat([
    pd.read_csv(f) for f in glob(os.path.join(PATH_TO_STORE, TARGET_SUBREDDIT, 'awards_*.csv'))
])

df['time_since_awarded'] = pd.Timestamp.utcnow() - pd.to_datetime( df.awarding_timestamp, unit='s', utc=True )

num_points = df.groupby( 'awarded_author' ).agg({'awarded_id':'count'}).iloc[:,0]
num_points = num_points.rename('num_critiquepoints')
print(f'Loaded {num_points.sum():,d} critiquepoints from {len(num_points):,d} users.')

post_delay = df.groupby( 'awarded_author' ).agg({'time_since_awarded':'min'}).iloc[:,0]
post_delay = post_delay.rename('time_since_last_award')

if FLAIR_UPDATE_EXCLUDE_POINTS_OLDER_THAN_HOURS is not None:
    users_to_update = post_delay < pd.Timedelta(FLAIR_UPDATE_EXCLUDE_POINTS_OLDER_THAN_HOURS, unit='hours')
    # add the force update users
    for user in bonus_flair_users.loc[bonus_flair_users.force_update,'username'].to_list():
         if user not in users_to_update.index:
                users_to_update[user] = True
    num_points = num_points.reindex( users_to_update.index[users_to_update] ).fillna(0)
    print(f"Updating {users_to_update.sum():,d} users' flair ", end='')
    print(f"who have received points in the last {FLAIR_UPDATE_EXCLUDE_POINTS_OLDER_THAN_HOURS} hours...")

else:
    print(f"Updating ALL {len(num_points):,d} users' flair...")


    

for i, (username, score) in enumerate(num_points.items()):
    flair_text = f"{score:,d} CritiquePoint{'s' if score>1 else ''}"
    
    if username in bonus_flair_set:
        flair_text = '★ | '+flair_text
    
    print(f"({(i/len(num_points))*100:5.1f}%) Setting {username}'s flair to `{flair_text}`")

    print('Finished.')



Loaded 1 users who get bonus flair for participating in the Coach user study.
Loaded 10,446 critiquepoints from 4,558 users.
Updating 41 users' flair who have received points in the last 48 hours...
(  0.0%) Setting 42tooth_sprocket's flair to `13 CritiquePoints`
Finished.
(  2.4%) Setting Artver's flair to `15 CritiquePoints`
Finished.
(  4.9%) Setting DimestoreAnselAdams's flair to `5 CritiquePoints`
Finished.
(  7.3%) Setting DinJarrus's flair to `7 CritiquePoints`
Finished.
(  9.8%) Setting Estemar20's flair to `1 CritiquePoint`
Finished.
( 12.2%) Setting Fogjazz62's flair to `1 CritiquePoint`
Finished.
( 14.6%) Setting Interesting-Suit7841's flair to `1 CritiquePoint`
Finished.
( 17.1%) Setting LegalBodybuilder5823's flair to `1 CritiquePoint`
Finished.
( 19.5%) Setting Lens_cap07's flair to `1 CritiquePoint`
Finished.
( 22.0%) Setting Misdirects's flair to `6 CritiquePoints`
Finished.
( 24.4%) Setting Miserable-Glass4084's flair to `1 CritiquePoint`
Finished.
( 26.8%) Setting Mis

In [3]:
num_points.isna().sum()

0

In [4]:
'cyclistNerd' in bonus_flair_users.username.to_list()

True

In [5]:
pd.read_csv('/projects/bdata/photocritique-coach-extension/logs/participants.csv')

,username,email,enrollmentTimestamp,assignedInterface
0,burly-gurly,NaN,2025-09-13T07:47:10.446Z,tutor
1,lew_traveler,llorton@gmail.com,2025-09-14T17:06:40.348Z,tutor
2,_Scorpion_1,adam.tencer@gmail.com,2025-09-14T19:40:41.129Z,control
3,leshlon,malikquess@gmail.com,2025-09-15T07:21:55.390Z,tutor
4,Normal-Armadillo8961,delphiniser@gmail.com,2025-09-23T08:07:06.908Z,tutor
5,noahmaier,maiernoah@gmail.com,2025-09-24T00:47:28.834Z,assistant
6,frootyFYt,drekoray09@gmail.com,2025-09-25T11:04:10.054Z,assistant
7,cyclistNerd,NaN,2025-11-04T21:28:33.593Z,control
